# Network Embedding

In [1]:
import os, sys
from pathlib import Path
from typing import List, Dict, Set
from collections import defaultdict
import random
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from tqdm import tqdm

DATA_PATH = Path("../data/FB15k-237/")
DATA_FILE = ["train.txt", "valid.txt", "test.txt"]

In [2]:
def get_triplets(path: Path) -> List:
    tsv = pd.read_csv(path, header=None, sep='\t')
    return list(tsv.itertuples(index=False, name=None))

train_triplets = get_triplets(DATA_PATH / DATA_FILE[0])
valid_triplets = get_triplets(DATA_PATH / DATA_FILE[1])
test_triplets = get_triplets(DATA_PATH / DATA_FILE[2])

triplets = train_triplets + valid_triplets + test_triplets
entity2id = {}
relation2id = {}

for h, r, t in triplets:
    entity2id.setdefault(h, len(entity2id))
    entity2id.setdefault(t, len(entity2id))
    relation2id.setdefault(r, len(relation2id))

id2entity = {v:k for k, v in entity2id.items()}
id2relation = {v:k for k, v in relation2id.items()}
num_entities = len(id2entity)
num_relations = len(id2relation)

print(f"Train Triplets: {len(train_triplets)}")
print(f"Valid Triplets: {len(valid_triplets)}")
print(f"Test Triplets: {len(test_triplets)}")
print(f"# of Entities: {num_entities}")
print(f"# of Relations: {num_relations}")

Train Triplets: 272115
Valid Triplets: 17535
Test Triplets: 20466
# of Entities: 14541
# of Relations: 237


## Random walk-based Methods

### Node2Vec

In [ ]:
# undirected
def build_adjacency(triplets, entity2id) -> Dict[int, List[int]]:
    adj = defaultdict(set)
    for h, r, t in triplets:
        h_id = entity2id[h]
        t_id = entity2id[t]
        adj[h_id].add(t_id)
        adj[t_id].add(h_id)

    return {node: list(neighbors) for node, neighbors in adj.items()}

adj = build_adjacency(train_triplets, entity2id)
print(f"Nodes with edges: {len(adj)}")
print(f"Avg degree: {np.mean([len(v) for v in adj.values()]):.1f}")

Nodes with edges: 14505
Avg degree: 29.2


In [11]:
def node2vec_walk(adj: Dict, start: int, walk_length: int,
                  p: float=1.0, q: float=1.0) -> List[int]:
    walk = [start]
    
    if start not in adj or len(adj[start]) == 0:
        return walk

    walk.append(random.choice(adj[start]))

    for _ in range(walk_length - 2):
        prev = walk[-2]
        cur = walk[-1]

        if cur not in adj or len(adj[cur]) == 0:
            break

        prev_neighbors = set(adj.get(prev, []))
        cur_neighbors = adj[cur]

        weights = []
        for nbr in cur_neighbors:
            if nbr == prev:
                weights.append(1.0 / p)
            elif nbr in prev_neighbors:
                weights.append(1.0)	# BFS
            else:
                weights.append(1.0 / q)

        total = sum(weights)
        probs = [w / total for w in weights]
        next_node = random.choices(cur_neighbors, weights=probs, k=1)[0]
        walk.append(next_node)

    return walk


def generate_walks(adj: Dict, num_entities: int, num_walks: int=10,
                   walk_length: int=80, p: float=1.0, q: float=1.0) -> List[List[int]]:
    walks = []
    nodes = list(range(num_entities))

    for _ in tqdm(range(num_walks), desc="Generating Walks"):
        random.shuffle(nodes)
        for node in nodes:
            if node in adj:
                walk = node2vec_walk(adj, node, walk_length, p, q)
                if len(walk) > 1:
                    walks.append(walk)

    print(f"\nTotal walks: {len(walks)}")
    return walks

walks = generate_walks(adj, num_entities, num_walks=10, walk_length=80, p=1.0, q=1.0)

Generating Walks:   0%|                                  | 0/10 [00:00<?, ?it/s]

Generating Walks:  10%|██▌                       | 1/10 [00:26<04:02, 26.91s/it]

Generating Walks:  20%|█████▏                    | 2/10 [00:54<03:37, 27.20s/it]

Generating Walks:  30%|███████▊                  | 3/10 [01:22<03:12, 27.54s/it]

Generating Walks:  40%|██████████▍               | 4/10 [01:50<02:46, 27.78s/it]

...

Generating Walks:  80%|████████████████████▊     | 8/10 [03:44<00:56, 28.35s/it]

Generating Walks:  90%|███████████████████████▍  | 9/10 [04:13<00:28, 28.53s/it]

Generating Walks: 100%|█████████████████████████| 10/10 [04:41<00:00, 28.61s/it]

Generating Walks: 100%|█████████████████████████| 10/10 [04:41<00:00, 28.19s/it]

Total walks: 145050


In [12]:
class SkipGramDataset(Dataset):
    def __init__(self, walks: List[List[int]], window_size: int=5):
        self.pairs = []
        for walk in tqdm(walks, desc="Building pairs"):
            for i, center in enumerate(walk):
                left = max(0, i - window_size)
                right = min(len(walk), i + window_size + 1)
                for j in range(left, right):
                    if i != j:
                        self.pairs.append((center, walk[j]))

    def __len__(self):
        return len(self.pairs)

    def __getitem__(self, idx):
        center, context = self.pairs[idx]
        return torch.tensor(center, dtype=torch.long), \
            torch.tensor(context, dtype=torch.long)

skipgram_dataset = SkipGramDataset(walks, window_size=5)
skipgram_loader = DataLoader(skipgram_dataset, batch_size=4096, shuffle=True)
print(f"\nTotal (center, context) pairs: {len(skipgram_dataset)}")

Building pairs:   0%|                                | 0/145050 [00:00<?, ?it/s]

Building pairs:   1%|▎                 | 2072/145050 [00:00<00:06, 20712.96it/s]

Building pairs:   3%|▌                 | 4792/145050 [00:00<00:05, 24522.81it/s]

Building pairs:   5%|▉                 | 7456/145050 [00:00<00:05, 25485.08it/s]

Building pairs:   7%|█▏               | 10146/145050 [00:00<00:05, 26038.61it/s]

...

Building pairs:  96%|███████████████▎| 139054/145050 [00:05<00:00, 26238.21it/s]

Building pairs:  98%|███████████████▋| 141771/145050 [00:05<00:00, 26515.94it/s]

Building pairs: 100%|███████████████▉| 144424/145050 [00:05<00:00, 26474.31it/s]

Building pairs: 100%|████████████████| 145050/145050 [00:05<00:00, 25968.38it/s]

Total (center, context) pairs: 111688500


In [13]:
class SkipGramModel(nn.Module):
    def __init__(self, num_nodes, emb_dim=100):
        super().__init__()
        self.emb_dim = emb_dim
        self.center_emb = nn.Embedding(num_nodes, emb_dim)
        self.context_emb = nn.Embedding(num_nodes, emb_dim)

        nn.init.xavier_uniform_(self.center_emb.weight)
        nn.init.xavier_uniform_(self.context_emb.weight)

    def forward(self, center, context, neg_context):
        c_emb = self.center_emb(center)
        ctx_emb = self.context_emb(context)
        neg_emb = self.context_emb(neg_context)

        pos_score = (c_emb * ctx_emb).sum(dim=1)
        pos_loss = -torch.log(torch.sigmoid(pos_score) + 1e-8).mean()

        neg_score = torch.bmm(neg_emb, c_emb.unsqueeze(2)).squeeze(2)
        neg_loss = -torch.log(torch.sigmoid(-neg_score) + 1e-8).mean()

        return pos_loss + neg_loss

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = SkipGramModel(num_entities, emb_dim=100).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
NUM_NEGATIVE = 5

EPOCHS = 10
for epoch in range(1, EPOCHS+1):
    model.train()
    total_loss = 0
    pbar = tqdm(skipgram_loader, desc=f"Epoch {epoch}", leave=True)

    for center, context in pbar:
        center = center.to(device)
        context = context.to(device)
        neg_context = torch.randint(0, num_entities,
                                    (center.size(0), NUM_NEGATIVE),
                                    device=device)

        optimizer.zero_grad()
        loss = model(center, context, neg_context)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        pbar.set_postfix(loss=loss.item())

    avg_loss = total_loss / len(skipgram_loader)
    print(f"Epoch {epoch} - Avg Loss: {avg_loss:.4f}")

def evaluate(model, triplets, all_triplets, entity2id, relation2id,
             num_entities, device, k_list=[1, 3, 10]):
    model.eval()
    ranks = []

    node_emb = model.center_emb.weight.data
    
    true_tails = {}
    for h, r, t in all_triplets:
        h_id = entity2id[h]; r_id = relation2id[r]; t_id = entity2id[t]
        true_tails.setdefault((h_id, r_id), set()).add(t_id)

    eval_data = [(entity2id[h], relation2id[r], entity2id[t]) for h, r, t in triplets]

    with torch.no_grad():
        for h_id, r_id, t_id in tqdm(eval_data, desc="Evaluating"):
            h_emb = node_emb[h_id].unsqueeze(0)

            scores = torch.norm(h_emb - node_emb, p=2, dim=1)

            filter_mask = torch.zeros(num_entities, dtype=torch.bool, device=device)
            for true_t in true_tails.get((h_id, r_id), set()):
                if true_t != t_id:
                    filter_mask[true_t] = True

            scores[filter_mask] = float("inf")

            correct_score = scores[t_id]
            rank = (scores < correct_score).sum().item() + 1
            ranks.append(rank)

    ranks = np.array(ranks)
    mrr = np.mean(1.0 / ranks)
    print(f"MRR: {mrr:.4f}")
    for k in k_list:
        hits = np.mean(ranks <= k)
        print(f"Hits@{k}: {hits:.4f}")

evaluate(model, valid_triplets, triplets, entity2id, relation2id, num_entities, device)

Epoch 1:   0%|                                        | 0/27268 [00:00<?, ?it/s]

Epoch 1:   0%|                             | 0/27268 [00:03<?, ?it/s, loss=1.39]

Epoch 1:   0%|                  | 1/27268 [00:03<23:22:20,  3.09s/it, loss=1.39]

Epoch 1:   0%|                  | 1/27268 [00:03<23:22:20,  3.09s/it, loss=1.39]

Epoch 1:   0%|                  | 1/27268 [00:03<23:22:20,  3.09s/it, loss=1.39]

...

Epoch 1:  27%|████           | 7452/27268 [54:12<1:32:15,  3.58it/s, loss=0.837]

Epoch 1:  27%|████▎           | 7452/27268 [54:13<1:32:15,  3.58it/s, loss=0.84]

Epoch 1:  27%|████           | 7452/27268 [54:13<1:32:15,  3.58it/s, loss=0.834]

Epoch 1:  27%|████           | 7454/27268 [54:13<1:10:24,  4.69it/s, loss=0.834]

Epoch 1:  27%|████           | 7454/27268 [54:13<1:10:24,  4.69it/s, loss=0.832]

## Graph Neural Networks

### GCN

In [5]:
def build_sparse_adj(triplets, entity2id, num_entities, device):
    edges = set()
    for h, r, t in triplets:
        h_id = entity2id[h]
        t_id = entity2id[t]
        edges.add((h_id, t_id))
        edges.add((t_id, h_id))

    for i in range(num_entities):
        edges.add((i, i))

    row = [e[0] for e in edges]
    col = [e[1] for e in edges]
    indices = torch.tensor([row, col], dtype=torch.long, device=device)

    deg = torch.zeros(num_entities, device=device)
    for r, c in zip(row, col):
        deg[r] += 1.0

    deg_inv_sqrt = deg.pow(-0.5)
    deg_inv_sqrt[deg_inv_sqrt == float("inf")] = 0.0

    values = deg_inv_sqrt[indices[0]] * deg_inv_sqrt[indices[1]]
    adj = torch.sparse_coo_tensor(indices, values, (num_entities, num_entities))
    adj = adj.coalesce()
    
    torch.sparse.check_sparse_tensor_invariants.disable()
    
    return adj

In [6]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
adj_sparse = build_sparse_adj(train_triplets, entity2id, num_entities, device)
print(f"Adjacency: {adj_sparse.shape}, nnz={adj_sparse._nnz()}")

Adjacency: torch.Size([14541, 14541]), nnz=436433


In [7]:
class EdgeDataset(Dataset):
    def __init__(self, triplets, entity2id):
        self.edges = [
            (entity2id[h], entity2id[t])
            for h, r, t in triplets
        ]

    def __len__(self):
        return len(self.edges)

    def __getitem__(self, idx):
        h, t = self.edges[idx]
        return torch.tensor(h, dtype=torch.long), \
            torch.tensor(t, dtype=torch.long)

edge_dataset = EdgeDataset(train_triplets, entity2id)
edge_loader = DataLoader(edge_dataset, batch_size=2048, shuffle=True)

In [11]:
class GCNLayer(nn.Module):
    def __init__(self, in_dim, out_dim):
        super().__init__()
        self.linear = nn.Linear(in_dim, out_dim, bias=False)
        nn.init.xavier_uniform_(self.linear.weight)

    def forward(self, x, adj):
        h = self.linear(x)
        h = torch.sparse.mm(adj, h)
        return h

class GCNEncoder(nn.Module):
    def __init__(self, num_nodes, in_dim=100, hidden_dim=100, out_dim=100, dropout=0.1):
        super().__init__()
        self.node_features = nn.Embedding(num_nodes, in_dim)
        nn.init.xavier_uniform_(self.node_features.weight)

        self.gcn1 = GCNLayer(in_dim, hidden_dim)
        self.gcn2 = GCNLayer(hidden_dim, out_dim)
        self.dropout = nn.Dropout(dropout)

    def forward(self, adj):
        x = self.node_features.weight
        x = self.gcn1(x, adj)
        x = F.relu(x)
        x = self.dropout(x)
        x = self.gcn2(x, adj)
        return x

    def get_embedding(self, adj):
        with torch.no_grad():
            x = self.node_features.weight
            x = F.relu(self.gcn1(x, adj))
            x = self.gcn2(x, adj)
        
        return x

    
class LinkPrediction(nn.Module):
    def __init__(self, num_nodes, emb_dim=100, hidden_dim=100, dropout=0.1):
        super().__init__()
        self.encoder = GCNEncoder(num_nodes, emb_dim, hidden_dim, emb_dim, dropout)
        self.emb_dim = emb_dim

    def encode(self, adj):
        return self.encoder(adj)

    def decode(self, node_emb, h_idx, t_idx):
        h_emb = node_emb[h_idx]
        t_emb = node_emb[t_idx]
        return (h_emb * t_emb).sum(dim=1)

    def forward(self, adj, pos_h, pos_t, neg_h, neg_t):
        node_emb = self.encode(adj)

        pos_score = self.decode(node_emb, pos_h, pos_t)
        neg_score = self.decode(node_emb, neg_h, neg_t)

        pos_loss = F.binary_cross_entropy_with_logits(
            pos_score, torch.ones_like(pos_score)
        )
        neg_loss = F.binary_cross_entropy_with_logits(
            neg_score, torch.ones_like(neg_score)
        )

        return pos_loss + neg_loss

In [ ]:
model = LinkPrediction(num_entities, emb_dim=100, hidden_dim=100, dropout=0.1).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-5)
scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=50, gamma=0.5)

EPOCHS = 100
for epoch in range(1, EPOCHS + 1):
    model.train()
    total_loss = 0
    pbar = tqdm(edge_loader, desc=f"Epoch {epoch}", leave=True)
 
    for pos_h, pos_t in pbar:
        pos_h = pos_h.to(device)
        pos_t = pos_t.to(device)

        neg_h = pos_h.clone()
        neg_t = pos_t.clone()
        corrupt_head = torch.rand(len(pos_h)) < 0.5
        rand_ent = torch.randint(0, num_entities, (len(pos_h),), device=device)
        neg_h[corrupt_head] = rand_ent[corrupt_head]
        neg_t[~corrupt_head] = rand_ent[~corrupt_head]
 
        optimizer.zero_grad()
        loss = model(adj_sparse, pos_h, pos_t, neg_h, neg_t)
        loss.backward()
        optimizer.step()
 
        total_loss += loss.item()
        pbar.set_postfix(loss=loss.item())
 
    avg_loss = total_loss / len(edge_loader)
    scheduler.step()
    print(f"Epoch {epoch} — Avg Loss: {avg_loss:.4f}")

def evaluate(model, adj, triplets, all_triplets, entity2id, relation2id,
                 num_entities, device, k_list=[1, 3, 10]):
    model.eval()
    ranks = []
    
    node_emb = model.encoder.get_embeddings(adj)
 
    true_tails = {}
    for h, r, t in all_triplets:
        h_id = entity2id[h]; r_id = relation2id[r]; t_id = entity2id[t]
        true_tails.setdefault((h_id, r_id), set()).add(t_id)
 
    eval_data = [(entity2id[h], relation2id[r], entity2id[t]) for h, r, t in triplets]
 
    with torch.no_grad():
        for h_id, r_id, t_id in tqdm(eval_data, desc="Evaluating"):
            h_emb = node_emb[h_id].unsqueeze(0)
 
            scores = (h_emb * node_emb).sum(dim=1)
 
            filter_mask = torch.zeros(num_entities, dtype=torch.bool, device=device)
            for true_t in true_tails.get((h_id, r_id), set()):
                if true_t != t_id:
                    filter_mask[true_t] = True
            scores[filter_mask] = float('-inf')
 
            correct_score = scores[t_id]
            rank = (scores > correct_score).sum().item() + 1
            ranks.append(rank)
 
    ranks = np.array(ranks)
    mrr = np.mean(1.0 / ranks)
    print(f"MRR: {mrr:.4f}")
    for k in k_list:
        hits = np.mean(ranks <= k)
        print(f"Hits@{k}: {hits:.4f}")


evaluate(model, adj_sparse, valid_triplets, triplets, entity2id, relation2id, num_entities, device)

Epoch 1:   0%|                                          | 0/133 [00:00<?, ?it/s]

Epoch 1:   0%|                               | 0/133 [00:00<?, ?it/s, loss=1.39]

Epoch 1:   0%|                               | 0/133 [00:00<?, ?it/s, loss=1.39]

Epoch 1:   2%|▎                      | 2/133 [00:00<00:08, 15.85it/s, loss=1.39]

Epoch 1:   2%|▎                      | 2/133 [00:00<00:08, 15.85it/s, loss=1.39]

...

Epoch 9:  23%|████▎              | 30/133 [00:01<00:04, 21.37it/s, loss=5.27e-5]

Epoch 9:  23%|████▎              | 30/133 [00:01<00:04, 21.37it/s, loss=4.41e-5]

Epoch 9:  23%|████▌               | 30/133 [00:01<00:04, 21.37it/s, loss=6.2e-5]

Epoch 9:  23%|████▎              | 30/133 [00:01<00:04, 21.37it/s, loss=3.76e-5]

Epoch 9:  25%|████▋              | 33/133 [00:01<00:04, 21.44it/s, loss=3.76e-5]